# Frozen v2 mechanistic rerun

Thin orchestration notebook for the final v2 GPU rerun on free-tier Colab T4.
It calls repository code (`src/analysis/v2_pipeline.py`, `src/analysis/v2_shards.py`,
`src/analysis/v2_compat.py`, `src/validate_benchmark_v2.py`) rather than
reimplementing batching, checkpointing, steering, statistics, or benchmark
logic here. It does not use the legacy mutable-eval-set pipeline.

Run cells top-to-bottom. Everything through the dry-run cell is safe to run
repeatedly and does not touch the GPU budget beyond a small calibration
pass. The live run is gated behind an explicit `RUN_GPU` flag.

Resumability: this notebook binds `results/` to a persistent Drive folder,
so a Colab disconnect or a fresh runtime does not lose completed shards -
just reopen this notebook and run top-to-bottom again; finished stages and
shards are skipped automatically (`src/analysis/v2_shards.py`,
`src/analysis/v2_pipeline.py`'s stage-major runner).


## 1. Mount Drive

Needed before persistent storage paths (below) can be created.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Clone and pin the exact commit

`PINNED_COMMIT` must be set to the final integration commit (after Milestones
2C/2D, 5A, and 6A land alongside this notebook change) before a live run.
Leaving it unset intentionally blocks the live-run cell later - it does not
block validation/dry-run, which only need *a* consistent checkout.

In [ ]:
import os
import subprocess

REPO_URL = 'https://github.com/urosavurdic/dpo-safety-representations.git'
REPO_DIR = '/content/dpo-safety-representations'
BRANCH = 'main'
PINNED_COMMIT = '37c22f4fa0d8a1098017f5518cdeb0b7ad4cf5dd'

if not os.path.exists(REPO_DIR):
    subprocess.run([
        'git', 'clone', '-b', BRANCH,
        REPO_URL, REPO_DIR,
    ], check=True)

os.chdir(REPO_DIR)
subprocess.run(['git', 'fetch', 'origin'], check=True)
subprocess.run([
    'git', 'checkout', BRANCH,
], check=True)
subprocess.run([
    'git', 'pull', '--ff-only', 'origin', BRANCH,
], check=True)

commit = subprocess.check_output([
    'git', 'rev-parse', 'HEAD',
], text=True).strip()
print('Checked out commit:', commit)


## 3. Persistent storage

Everything that must survive a Colab disconnect or a fresh runtime is
bound under one Drive root:

- `results/` (activations, direction vectors, behavioral output, shard
  checkpoints/progress) - what `src/analysis/v2_pipeline.py` and
  `src/analysis/v2_shards.py` read and write by their normal relative
  path. Bound by replacing the local `results/` with a symlink into Drive,
  so no pipeline code changes; this is storage plumbing, not pipeline logic.
- Hugging Face cache (base model + LoRA adapters, pulled from the Hub by
  `src/training/model.py`'s `STAGE_ADAPTER_CHAINS` on every stage load) -
  bound via `HF_HOME`, set before any `transformers`/`peft` import happens,
  so weights are downloaded once total across every session, not once per
  session.

`logs/*.json` (calibration, split manifest, validation status) are cheap,
deterministic, and regenerated from committed inputs each session, so they
are intentionally left on local (ephemeral) disk - only `results/` and the
model cache need to survive a runtime restart.

Nothing here is ever committed to Git - Drive is the only persistence
mechanism for these artifacts (see the `.gitignore` rules already in this
repo for why: `results/activations/*.npy` and friends are large and
deliberately untracked).

In [ ]:
import shutil
from pathlib import Path

# Change this if you want multiple independent rerun attempts side by side;
# otherwise every Colab session for this branch reattaches to the same state.
DRIVE_ROOT = Path('/content/drive/MyDrive/dpo_safety_v2')
DRIVE_RESULTS = DRIVE_ROOT / 'results'
DRIVE_HF_CACHE = DRIVE_ROOT / 'hf_cache'

DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
DRIVE_HF_CACHE.mkdir(parents=True, exist_ok=True)

# Must be set before any transformers/peft import - src/training/model.py's
# load_stage_model() always calls .from_pretrained() against the Hub, and an
# unset HF_HOME would silently re-download the ~3 GB base model (plus LoRA
# adapters) on every single stage load, every session.
os.environ['HF_HOME'] = str(DRIVE_HF_CACHE)

local_results = Path('results')
if local_results.is_symlink():
    pass  # already wired up - e.g. this cell re-run mid-session
elif not (DRIVE_RESULTS / 'activations').exists():
    # First use of this Drive root: seed it with what the checkout already
    # ships (the small, pre-v2 committed artifacts under results/) so the v2
    # pipeline layers new v2-suffixed output on top instead of starting from
    # an empty directory. src/analysis/v2_compat.py depends on these
    # pre-v2 files being present and untouched until it explicitly bridges
    # v2 output over them.
    print(f'Seeding {DRIVE_RESULTS} from the checkout\'s committed results/ ...')
    shutil.copytree(local_results, DRIVE_RESULTS, dirs_exist_ok=True)
    shutil.rmtree(local_results)
    local_results.symlink_to(DRIVE_RESULTS, target_is_directory=True)
else:
    # Resuming: Drive already holds a prior session's progress (shard
    # checkpoints, partial activations, ...). Discard the fresh clone's copy
    # (identical to what was already seeded) and point at Drive instead.
    shutil.rmtree(local_results)
    local_results.symlink_to(DRIVE_RESULTS, target_is_directory=True)

print('Resolved persistent paths:')
print(f'  results/ -> {local_results.resolve()}')
print(f'  HF_HOME  -> {os.environ["HF_HOME"]}')


## 4. Install dependencies, check GPU

In [ ]:
!python -m pip install -q -r requirements.txt
!python -m pip uninstall -y torchao || true


In [ ]:
# GPU availability (required item: this notebook does not itself change the
# Colab runtime type - if this warns, use Runtime > Change runtime type > T4
# GPU, then re-run this cell).
gpu_check = subprocess.run(
    ['python', '-c',
     "import torch; print('CUDA available:', torch.cuda.is_available()); "
     "print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')"],
    capture_output=True, text=True,
)
print(gpu_check.stdout)
if gpu_check.returncode != 0:
    print('STDERR:', gpu_check.stderr)
if 'CUDA available: True' not in gpu_check.stdout:
    print('WARNING: no GPU detected. Calibration/live-run cells below need one.')


In [ ]:
print('=== python -m compileall src ===')
compileall_run = subprocess.run(
    ['python', '-m', 'compileall', 'src'],
    capture_output=True, text=True,
)
print(compileall_run.stdout[-2000:])
if compileall_run.returncode != 0:
    print('STDERR:', compileall_run.stderr[-2000:])
    raise RuntimeError('src/ does not even compile - stop here.')


## 5. Benchmark, split-manifest, and gate verification

Regenerates the direction split manifest and re-validates the frozen
benchmark through the repository's own scripts, then checks the static
gate fields the live run itself will enforce (`gate_for_run` in
`src/analysis/v2_pipeline.py`) - imported here rather than re-listed, so
this check can never silently drift from what the run cell actually
enforces a few cells down.

`artifact_freshness_pass` is expected to read `False` until this notebook's
own GPU pass produces fresh activations - that is what the live run cell
is for, not a problem to fix here.

In [ ]:
import json

from src.analysis.v2_pipeline import STATIC_GATE_FIELDS

latest = json.load(open('data/frozen_v2/LATEST_BENCHMARK.json'))
bench = latest['benchmark_path']

subprocess.run([
    'python', '-m',
    'src.create_direction_split_manifest',
    '--benchmark', bench,
], check=True)

subprocess.run([
    'python', '-m',
    'src.validate_benchmark_v2',
    '--benchmark', bench,
    '--review-csv',
    'data/review/c_review_queue.csv',
    '--gate-config',
    'logs/benchmark_gate_config.json',
    '--split-manifest',
    'logs/direction_split_manifest.json',
], check=True)

status = json.load(open(
    'logs/benchmark_validation_status.json'
))
assert all(status.get(k) is True for k in STATIC_GATE_FIELDS), status
print('Static benchmark checks passed:', STATIC_GATE_FIELDS)
print('Benchmark SHA-256:', status['benchmark_sha256'])
print('Split manifest SHA-256:', status['inputs']['split_manifest_sha256'])
print('technical_benchmark_status:', status['technical_benchmark_status'])
print(
    'artifact_freshness_pass:', status['artifact_freshness_pass'],
    '(expected False before this session\'s GPU pass produces fresh activations)',
)


## 6. Focused test gate

Runs after the benchmark itself has been verified above, so a benchmark-integrity failure surfaces at step 5 and is never masked by (or confused with) a code-test failure here. `V2_TEST_SCOPE` below is the lightweight, v2/notebook-relevant scope - not the full `pytest tests/` suite; see the comment in the cell for why.

In [ ]:
# Lightweight, v2/notebook-relevant test scope - NOT the full `pytest tests/`
# suite. The full suite also exercises training/ and data_pipeline/ modules
# outside this milestone's scope, and two tests that need live network
# access (documented in their own source): a sentence-transformers model
# download in tests/diagnostics/test_check_within_eval_set_dedup.py, and a
# StrongREJECT CSV fetch in tests/data_pipeline/test_quadrant_c_pipeline.py.
# Gating every Colab session's startup on those is exactly the kind of
# unreliable-optional-dependency gate this milestone should avoid. This
# scope is every test that actually protects the code path this notebook
# drives, plus core-import/environment sanity.
V2_TEST_SCOPE = [
    'tests/test_environment.py',
    'tests/test_v2_io.py',
    'tests/test_validate_benchmark_v2.py',
    'tests/test_config_consistency.py',
    'tests/analysis/test_v2_shards.py',
    'tests/analysis/test_v2_pipeline.py',
    'tests/analysis/test_v2_pipeline_deadline.py',
    'tests/analysis/test_v2_pipeline_steering.py',
    'tests/analysis/test_v2_resumability.py',
    'tests/analysis/test_v2_compat.py',
    'tests/analysis/test_v2_direction_family.py',
]

test_run = subprocess.run(
    ['python', '-m', 'pytest', *V2_TEST_SCOPE, '-q'],
    capture_output=True, text=True,
)
print(test_run.stdout[-4000:])
if test_run.returncode != 0:
    print('STDERR:', test_run.stderr[-2000:])
    raise RuntimeError(
        'v2-relevant tests failed. This scope excludes the modules known to '
        'need network access, so a failure here is a genuine problem, not '
        'an environment gap - stop and investigate before proceeding.'
    )
print('v2-relevant lightweight tests: all passed.')
print(
    '\n(Full suite `pytest tests/ -v` is also available for a complete pass, '
    'but is not run automatically here - see the note above.)'
)


## 7. Current progress

Whatever is already on persistent `results/` from a prior session, via the repository's own status command.

In [ ]:
!python -m src.analysis.v2_pipeline status


## 8. Throughput calibration

T4 timing varies enough between sessions that a hard-coded batch size or
session-count estimate is worse than useless (see
`src/analysis/v2_pipeline.py`'s `cmd_calibrate` docstring) - so this runs
every session. `--probe-capacity` additionally measures the largest
forward/generation batch size this GPU survives without OOM and records
`recommended_act_batch`/`recommended_gen_batch`, which `--act-batch auto` /
`--gen-batch auto` below read back. This is a tiny startup check (one
stage, 32 prompts, plus a short doubling search), not the GPU experiment
itself.

In [ ]:
SESSION_DEADLINE_MINUTES = 300  # ~5:30 T4 session minus setup/install margin

calibrate_run = subprocess.run(
    ['python', '-m', 'src.analysis.v2_pipeline', 'calibrate',
     '--stage', 'M3', '--n-prompts', '32',
     '--probe-capacity',
     '--deadline-minutes', str(SESSION_DEADLINE_MINUTES)],
    capture_output=True, text=True,
)
print(calibrate_run.stdout)
if calibrate_run.returncode != 0:
    print('STDERR:', calibrate_run.stderr[-2000:])
    raise RuntimeError('Calibration failed - see stderr above.')


## 9. Dry run

Constructs the resumable v2 execution plan under the calibrated batch
sizes and the configured session deadline, without touching the model.
Safe to run repeatedly. `--regenerate` is required by the gate whenever
`technical_benchmark_status` is not `PASS` (expected pre-run, see step 5).
`--with-norm-diag` is included so the printed plan (and the live run in
step 10) covers the complete intended run graph - causal ablation on the
4 DPO endpoints, steering on every stage but M0, and the norm diagnostic
following steering - per Milestone 6A/6B's verification of the full
steering/norm-diagnostic integration.

In [ ]:
!bash rerun_mechanistic_v2.sh --dry-run --regenerate --with-probes --with-norm-diag --act-batch auto --gen-batch auto --deadline-minutes {SESSION_DEADLINE_MINUTES}


## 10. Live run

Gated behind `RUN_GPU`. Set it to `True` and re-run this cell to launch a
real session against persistent `results/` - completed shards from a prior
session are skipped automatically, and the run stops cleanly at a shard
boundary once `SESSION_DEADLINE_MINUTES` is spent. Invokes the v2 runner
only (`rerun_mechanistic_v2.sh` -> `python -m src.analysis.v2_pipeline
run`), never the obsolete legacy GPU scripts. Same flags as the dry run
in step 9 (including `--with-norm-diag`), so what you saw previewed is
exactly what executes.

Requires `PINNED_COMMIT` (step 2) to be set to the real final integration
commit.

In [ ]:
RUN_GPU = False

if RUN_GPU:
    assert PINNED_COMMIT != '37c22f4fa0d8a1098017f5518cdeb0b7ad4cf5dd', (
        'Set PINNED_COMMIT to the final integration commit before a live run.'
    )
    commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
    assert commit == PINNED_COMMIT, f'Wrong commit: {commit}'
    !bash rerun_mechanistic_v2.sh --regenerate --with-probes --with-norm-diag --act-batch auto --gen-batch auto --deadline-minutes {SESSION_DEADLINE_MINUTES}
else:
    print('RUN_GPU is False - the dry run above already showed the plan this '
          'would execute. Set RUN_GPU = True and re-run this cell to launch it.')


## 11. Post-run: bridge outputs, re-validate, session summary

`src/analysis/v2_compat.py` copies whatever v2-suffixed output currently
exists (possibly partial, if the deadline stopped the run mid-stage) to
the legacy filenames/fields the CPU statistics layer
(`src.reproduce`, `summarize_probe_findings.py`, `summarize_cross_branch.py`)
already reads - safe to call every session; it never overwrites a
pre-v2 committed artifact it did not itself produce.

In [ ]:
if RUN_GPU:
    print('=== Bridging v2 outputs to legacy names ===')
    compat_run = subprocess.run(
        ['python', '-m', 'src.analysis.v2_compat'],
        capture_output=True, text=True,
    )
    print(compat_run.stdout)
    if compat_run.returncode != 0:
        print('STDERR:', compat_run.stderr[-2000:])

    print('\n=== Re-validating the benchmark gate against fresh activations ===')
    subprocess.run([
        'python', '-m',
        'src.validate_benchmark_v2',
        '--benchmark', latest['benchmark_path'],
        '--review-csv',
        'data/review/c_review_queue.csv',
        '--gate-config',
        'logs/benchmark_gate_config.json',
        '--split-manifest',
        'logs/direction_split_manifest.json',
    ], check=True)
    status = json.load(open('logs/benchmark_validation_status.json'))
    print('technical_benchmark_status:', status['technical_benchmark_status'])
    if status['technical_benchmark_status'] != 'PASS':
        print(
            'Not PASS yet - expected if stages are still incomplete across '
            'sessions. Re-run this notebook top-to-bottom to continue; '
            'finished shards/stages are skipped automatically.'
        )


In [ ]:
print('=' * 70)
print('SESSION SUMMARY')
print('=' * 70)

commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print(f'Git commit:          {commit}')
print(f'Benchmark:           {latest["benchmark_path"]}')
print(f'Benchmark SHA-256:   {latest["benchmark_sha256"]}')
print(f'Persistent results:  {Path("results").resolve()}')
print(f'Persistent HF cache: {os.environ.get("HF_HOME")}')

print('\n--- Progress (results/) ---')
!python -m src.analysis.v2_pipeline status

print('\n--- Next action ---')
if not RUN_GPU:
    print('This session only validated, calibrated, and dry-ran the plan above.')
    print('Set RUN_GPU = True in the live-run cell (step 10) and re-run it to execute.')
else:
    print('Live run cell executed above. If the plan is not yet fully complete,')
    print('reopen this notebook in a fresh Colab session and run top-to-bottom -')
    print('completed shards/stages are skipped automatically; incomplete ones')
    print('resume from the first unfinished shard.')
